### Engenharia de Features

##### Nessa etapa, transformamos o que encontramos na última EDA em colunas numéricas que o modelo consiga utilizar.

In [15]:
import pandas as pd
import numpy as np

Carrega e converte a data

In [3]:
df_final = pd.read_csv("../data/processed/base_unificada.csv")

df_final["order_purchase_timestamp"] = pd.to_datetime(df_final["order_purchase_timestamp"])

Aqui, vamos construir a feature que a última EDA revelou dos pedidos por vendedor, a taxa histórica de atraso de cada um. Algumas exceções precisaram ser tratadas pra evitar vazamento.


In [4]:
pedidos_vendedor = df_final[["order_id","seller_id","order_purchase_timestamp","dias_atrasados"]].drop_duplicates("order_id")
pedidos_vendedor = pedidos_vendedor.sort_values("order_purchase_timestamp").reset_index(drop=True)

pedidos_vendedor["atrasado_flag"] = (pedidos_vendedor["dias_atrasados"] > 0).astype(int)

pedidos_vendedor["pedidos_anteriores"] = pedidos_vendedor.groupby("seller_id").cumcount()
pedidos_vendedor["atrasos_anteriores"] = pedidos_vendedor.groupby("seller_id")["atrasado_flag"].transform(lambda x: x.shift().cumsum())

pedidos_vendedor["taxa_atraso_vendedor"] = pedidos_vendedor["atrasos_anteriores"] / pedidos_vendedor["pedidos_anteriores"]

Só junta de volta a taxa calculada na base principal, cruzando por order_id 

In [5]:
df_final = df_final.merge(pedidos_vendedor[["order_id","taxa_atraso_vendedor"]], on="order_id", how="left")

Confirmamos o NaN para validar que a feature não vazou, esses 3499 são os primeiros pedidos de cada vendedor que ainda não possuem histórico anterior.

In [6]:
df_final["taxa_atraso_vendedor"].isna().sum()

np.int64(3499)

Salvando o resultado com essa nova feature, já pronto pra entrar na modelagem

In [7]:
df_final.to_csv("../data/processed/df_features.csv", index=False)

-----------------------------------------------------------------------------

Agora, vamos fazer a geolocalização e cruzar por CEP.

Pelo que identificamos antes na EDA, cada CEP tem coordenadas diferentes (ou seja, são exatamente os pontos dentro do CEP, onde estão localizados. Uma abordagem ok, que quero testar, é calcular a média de todos esses pontos dentro de cada CEP, ou seja, agrupar cada número de CEP em um groupby, depois dentro de cada groupby, termos uma média de todos os pontos que servirá como padrão para a análise.)

In [13]:
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

geolocation_media = geolocation.groupby("geolocation_zip_code_prefix")[["geolocation_lat","geolocation_lng"]].mean() 
geolocation_media = geolocation_media.reset_index()

Primeiro, vamos dar merge no dataset final. O dataset recebe as informações de onde está o CEP do cliente, cruzando customer_zip_code_prefix e geolocation_zip_code_prefix

Depois, cruza seller_zip_code_prefix e geolocation_zip_code_prefix, que cruza com o CEP do vendedor.

É basicamente pra obtermos o CEP de ambos: vendedor e cliente.

In [14]:
df_final = df_final.merge(geolocation_media, left_on="customer_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")
df_final = df_final.merge(geolocation_media, left_on="seller_zip_code_prefix", right_on="geolocation_zip_code_prefix", how="left")

df_final

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,dias_atrasados,order_item_id,...,review_score,payment_type,payment_installments,taxa_atraso_vendedor,geolocation_zip_code_prefix_x,geolocation_lat_x,geolocation_lng_x,geolocation_zip_code_prefix_y,geolocation_lat_y,geolocation_lng_y
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-7.107488,1,...,4.0,credit_card,1.0,0.000000,3149.0,-23.576983,-46.587161,9350.0,-23.680729,-46.444238
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-5.355729,1,...,4.0,boleto,1.0,0.018182,47813.0,-12.177924,-44.660711,31570.0,-19.807681,-43.980427
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-17.245498,1,...,5.0,credit_card,3.0,0.119292,75265.0,-16.745150,-48.514783,14840.0,-21.363502,-48.229601
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-12.980069,1,...,5.0,credit_card,1.0,0.027027,59296.0,-5.774190,-35.271143,31842.0,-19.837682,-43.924053
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-9.238171,1,...,5.0,credit_card,1.0,0.119048,9195.0,-23.676370,-46.514627,8752.0,-23.543395,-46.262086
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110184,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,-1.265324,1,...,4.0,credit_card,3.0,0.027027,11722.0,-24.001500,-46.449864,17602.0,-21.930548,-50.498348
110185,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,-5.524803,1,...,5.0,credit_card,5.0,0.060000,45920.0,-17.898358,-39.373630,8290.0,-23.553642,-46.452661
110186,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,-20.018819,1,...,2.0,credit_card,4.0,0.043478,28685.0,-22.562825,-42.694574,37175.0,-20.940578,-45.827237
110187,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,-20.018819,2,...,2.0,credit_card,4.0,0.043478,28685.0,-22.562825,-42.694574,37175.0,-20.940578,-45.827237


Calculando a distância de Haversine (a escolha é por conta das features de latitude e longitude. Como essas medidas geográficas  se referem ao globo, não faz sentido comparar com uma distância reta). Foi uma das soluções que encontramos e vamos utilizar isso mesmo.

Precisamos usar o numpy pra isso, pra transformar em radianos. Como o cálculo utiliza valores trigonométricos, só dá pra utilizar radianos, ao invés de apenas os valores que tínhamos antes.

In [16]:
def calcula_distancia(lat1, lng1, lat2, lng2):
    lat1_rad = np.radians(lat1)
    lat2_rad = np.radians(lat2)
    delta_lat = np.radians(lat2 - lat1)
    delta_lng = np.radians(lng2 - lng1)

    a = np.sin(delta_lat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(delta_lng/2)**2
    c = 2 * np.arcsin(np.sqrt(a))

    return 6371 * c

Aplicando a função

In [17]:
df_final["distancia_km"] = calcula_distancia(df_final["geolocation_lat_x"], df_final["geolocation_lng_x"], df_final["geolocation_lat_y"], df_final["geolocation_lng_y"])

Compara se o vendedor reside no mesmo estado do cliente, e retorna True ou False.

In [18]:
df_final["cross_state"] = df_final["customer_state"] != df_final["seller_state"]

In [20]:
df_final

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,dias_atrasados,order_item_id,...,payment_installments,taxa_atraso_vendedor,geolocation_zip_code_prefix_x,geolocation_lat_x,geolocation_lng_x,geolocation_zip_code_prefix_y,geolocation_lat_y,geolocation_lng_y,distancia_km,cross_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,-7.107488,1,...,1.0,0.000000,3149.0,-23.576983,-46.587161,9350.0,-23.680729,-46.444238,18.576110,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,-5.355729,1,...,1.0,0.018182,47813.0,-12.177924,-44.660711,31570.0,-19.807681,-43.980427,851.495069,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,-17.245498,1,...,3.0,0.119292,75265.0,-16.745150,-48.514783,14840.0,-21.363502,-48.229601,514.410666,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,-12.980069,1,...,1.0,0.027027,59296.0,-5.774190,-35.271143,31842.0,-19.837682,-43.924053,1822.226336,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,-9.238171,1,...,1.0,0.119048,9195.0,-23.676370,-46.514627,8752.0,-23.543395,-46.262086,29.676625,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110184,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02,-1.265324,1,...,3.0,0.027027,11722.0,-24.001500,-46.449864,17602.0,-21.930548,-50.498348,474.120037,False
110185,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27,-5.524803,1,...,5.0,0.060000,45920.0,-17.898358,-39.373630,8290.0,-23.553642,-46.452661,967.847297,True
110186,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,-20.018819,1,...,4.0,0.043478,28685.0,-22.562825,-42.694574,37175.0,-20.940578,-45.827237,370.404482,True
110187,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15,-20.018819,2,...,4.0,0.043478,28685.0,-22.562825,-42.694574,37175.0,-20.940578,-45.827237,370.404482,True


Alguns CEPs não existem na tabela geolocation, tanto de clientes, quanto de vendedores. Por isso, naturalmente, alguns valores vieram faltantes.

In [19]:
df_final["distancia_km"].isna().sum()

np.int64(536)

In [21]:
df_final.to_csv("../data/processed/df_features.csv", index=False)